In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv("../../data/melb_data.csv")

In [3]:
data.head()

,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,3/12/2016,2.5,3067.0,...,1.0,1.0,202.0,NaN,NaN,Yarra,-37.7996,144.9984,Northern Metropolitan,4019.0
1,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,4/02/2016,2.5,3067.0,...,1.0,0.0,156.0,79.0,1900.0,Yarra,-37.8079,144.9934,Northern Metropolitan,4019.0
2,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,4/03/2017,2.5,3067.0,...,2.0,0.0,134.0,150.0,1900.0,Yarra,-37.8093,144.9944,Northern Metropolitan,4019.0
3,Abbotsford,40 Federation La,3,h,850000.0,PI,Biggin,4/03/2017,2.5,3067.0,...,2.0,1.0,94.0,NaN,NaN,Yarra,-37.7969,144.9969,Northern Metropolitan,4019.0
4,Abbotsford,55a Park St,4,h,1600000.0,VB,Nelson,4/06/2016,2.5,3067.0,...,1.0,2.0,120.0,142.0,2014.0,Yarra,-37.8072,144.9941,Northern Metropolitan,4019.0


In [4]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 13580 entries, 0 to 13579
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Suburb         13580 non-null  str    
 1   Address        13580 non-null  str    
 2   Rooms          13580 non-null  int64  
 3   Type           13580 non-null  str    
 4   Price          13580 non-null  float64
 5   Method         13580 non-null  str    
 6   SellerG        13580 non-null  str    
 7   Date           13580 non-null  str    
 8   Distance       13580 non-null  float64
 9   Postcode       13580 non-null  float64
 10  Bedroom2       13580 non-null  float64
 11  Bathroom       13580 non-null  float64
 12  Car            13518 non-null  float64
 13  Landsize       13580 non-null  float64
 14  BuildingArea   7130 non-null   float64
 15  YearBuilt      8205 non-null   float64
 16  CouncilArea    12211 non-null  str    
 17  Lattitude      13580 non-null  float64
 18  Longtitude     13

In [5]:
data.isnull().sum()

Suburb              0
Address             0
Rooms               0
Type                0
Price               0
Method              0
SellerG             0
Date                0
Distance            0
Postcode            0
Bedroom2            0
Bathroom            0
Car                62
Landsize            0
BuildingArea     6450
YearBuilt        5375
CouncilArea      1369
Lattitude           0
Longtitude          0
Regionname          0
Propertycount       0
dtype: int64

In [6]:
selectColumns=['Car','Landsize','BuildingArea','YearBuilt','Price']
df_selected = data[selectColumns]

In [7]:
df_selected.corr()

,Car,Landsize,BuildingArea,YearBuilt,Price
Car,1.000000,0.026770,0.096101,0.104515,0.238979
Landsize,0.026770,1.000000,0.500485,0.036451,0.037507
BuildingArea,0.096101,0.500485,1.000000,0.019665,0.090981
YearBuilt,0.104515,0.036451,0.019665,1.000000,-0.323617
Price,0.238979,0.037507,0.090981,-0.323617,1.000000


### From the above Corr it shows that the data is not correlated to our label Price when checked with corr function, this would eliminate all the features, lets try with the next approach to see for best features

# Backward Elimination using OLS

In [8]:
features = data.iloc[:, [12,13,14,15]].values
labels = data.iloc[:, [4]].values

In [9]:
features

array([[1.000e+00, 2.020e+02,       nan,       nan],
       [0.000e+00, 1.560e+02, 7.900e+01, 1.900e+03],
       [0.000e+00, 1.340e+02, 1.500e+02, 1.900e+03],
       ...,
       [4.000e+00, 4.360e+02,       nan, 1.997e+03],
       [5.000e+00, 8.660e+02, 1.570e+02, 1.920e+03],
       [1.000e+00, 3.620e+02, 1.120e+02, 1.920e+03]], shape=(13580, 4))

In [10]:
from sklearn.impute import SimpleImputer

# Car

siCar = SimpleImputer(missing_values=np.nan, strategy='mean')
features[:, [0]] = siCar.fit_transform(features[:,[0]])

#BuildingArea
siBuildingArea=SimpleImputer(missing_values=np.nan, strategy='mean')

features[:,[2]]=siBuildingArea.fit_transform(features[:,[2]])
#YearBuild (As this is date we will use the Mode(most_frequent in sklearn startegy)  strategy)

siYearBuilt=SimpleImputer(missing_values=np.nan, strategy='most_frequent')

features[:,[3]]=siYearBuilt.fit_transform(features[:,[3]])

In [11]:
featuresAllIn = np.append(np.ones((len(features), 1)).astype(int), features, axis=1)
featuresAllIn

array([[1.0000000e+00, 1.0000000e+00, 2.0200000e+02, 1.5196765e+02,
        1.9700000e+03],
       [1.0000000e+00, 0.0000000e+00, 1.5600000e+02, 7.9000000e+01,
        1.9000000e+03],
       [1.0000000e+00, 0.0000000e+00, 1.3400000e+02, 1.5000000e+02,
        1.9000000e+03],
       ...,
       [1.0000000e+00, 4.0000000e+00, 4.3600000e+02, 1.5196765e+02,
        1.9970000e+03],
       [1.0000000e+00, 5.0000000e+00, 8.6600000e+02, 1.5700000e+02,
        1.9200000e+03],
       [1.0000000e+00, 1.0000000e+00, 3.6200000e+02, 1.1200000e+02,
        1.9200000e+03]], shape=(13580, 5))

In [12]:
SL=0.05

import statsmodels.regression.linear_model as stat

model = stat.OLS(endog=labels, exog=featuresAllIn).fit()
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.139
Model:                            OLS   Adj. R-squared:                  0.138
Method:                 Least Squares   F-statistic:                     546.5
Date:                Fri, 26 Jun 2026   Prob (F-statistic):               0.00
Time:                        18:53:37   Log-Likelihood:            -1.9979e+05
No. Observations:               13580   AIC:                         3.996e+05
Df Residuals:                   13575   BIC:                         3.996e+05
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       1.291e+07   3.45e+05     37.442      0.000    1.22e+07    1.36e+07
x1          1.711e+05   5332.865     32.077      0.000    1.61e+05    1.82e+05
x2             4.5330      1.282      3.536      0.000       2.020       7.046
x3            82.7988     13.078      6.331      0.000      57.164     108.433
x4         -6166.5913    175.637    -35.110      0.000   -6510.865   -5822.318
==============================================================================
Omnibus:                     6828.746   Durbin-Watson:                   1.468
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            77784.791
Skew:                           2.149   Prob(JB):                         0.00
Kurtosis:                      13.908   Cond. No.                     2.74e+05
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 2.74e+05. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

# Using RFE (Recursive Feature Elimination)

In [13]:
from sklearn.linear_model import LinearRegression
modelAlgo = LinearRegression()

In [14]:
from sklearn.feature_selection import RFE
selecteFeatures = RFE(estimator=modelAlgo)

In [15]:
selecteFeatures.fit(features, labels)

RFE(estimator=LinearRegression())

In [16]:
selecteFeatures.ranking_

array([1, 3, 2, 1])

### Guideline: Select the columns with ranking 1 and 2 based on this we can select Car and Building Area

In [17]:
features=data.iloc[:,[12,14,15]].values
label=data.iloc[:,[4]].values

In [18]:
#Car
siCar=SimpleImputer(missing_values=np.nan, strategy='mean')

features[:,[0]]=siCar.fit_transform(features[:,[0]])

#BuildingArea
siBuildingArea=SimpleImputer(missing_values=np.nan, strategy='mean')

features[:,[1]]=siBuildingArea.fit_transform(features[:,[1]])

#YearBuild (As this is date we will use the Mode(most_frequent in sklearn startegy)  strategy)

siYearBuilt=SimpleImputer(missing_values=np.nan, strategy='most_frequent')

features[:,[2]]=siYearBuilt.fit_transform(features[:,[2]])

In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test=train_test_split(features,
                                                 label,
                                                 test_size=0.2,
                                                 random_state=10)

model=LinearRegression()
model.fit(X_train, y_train)

trainScore=model.score(X_train, y_train)
testScore=model.score(X_test, y_test)

print(f"Test Score: {testScore} and Train Score : {trainScore}")

Test Score: 0.12490205728298187 and Train Score : 0.1405290457314029


In [20]:
CL=0.05

for seed in range(1,100):

    X_train, X_test, y_train, y_test=train_test_split(features,
                                                     label,
                                                     test_size=0.2,
                                                     random_state=seed)

    model=LinearRegression()
    model.fit(X_train, y_train)

    trainScore=model.score(X_train, y_train)
    testScore=model.score(X_test, y_test)

    if testScore > trainScore and testScore >= CL:

        print(f"Test Score: {testScore} and Train Score: {trainScore} for Random Seed: {seed}")

Test Score: 0.15451047349772784 and Train Score: 0.1334296507444903 for Random Seed: 6
Test Score: 0.16799581087305038 and Train Score: 0.13051025532335025 for Random Seed: 11
Test Score: 0.13832688100413526 and Train Score: 0.13720545162950815 for Random Seed: 12
Test Score: 0.14581190572484215 and Train Score: 0.13558821109750974 for Random Seed: 14
Test Score: 0.15611301417463275 and Train Score: 0.13346072102395423 for Random Seed: 15
Test Score: 0.16246074911310038 and Train Score: 0.13108964711051796 for Random Seed: 18
Test Score: 0.148636127520547 and Train Score: 0.13477561420421547 for Random Seed: 19
Test Score: 0.1510867474764901 and Train Score: 0.13426149523551023 for Random Seed: 20
Test Score: 0.14170933531794128 and Train Score: 0.1366041079630128 for Random Seed: 21
Test Score: 0.16245167634122704 and Train Score: 0.13137593872249542 for Random Seed: 22
Test Score: 0.1561884014227215 and Train Score: 0.13335277424136083 for Random Seed: 26
Test Score: 0.14320636737438

# Based on this score it shows that the model bit better or similar when selecting all the features and gives almost same score
## Test Score: 0.1702233396678955 and Train Score: 0.13038930544961236 for Random Seed: 57

## SFM suggest to consider Car

In [32]:
features=data.iloc[:,[12]].values
label=data.iloc[:,[4]].values

In [34]:
data.isna().sum()

Suburb              0
Address             0
Rooms               0
Type                0
Price               0
Method              0
SellerG             0
Date                0
Distance            0
Postcode            0
Bedroom2            0
Bathroom            0
Car                62
Landsize            0
BuildingArea     6450
YearBuilt        5375
CouncilArea      1369
Lattitude           0
Longtitude          0
Regionname          0
Propertycount       0
dtype: int64

In [38]:
#Car
siCar=SimpleImputer(missing_values=np.nan, strategy='mean')



In [36]:
features = data.iloc[:, [12]].to_numpy(copy=True)
siCar = SimpleImputer(missing_values=np.nan, strategy='mean')
features[:, [0]] = siCar.fit_transform(features[:, [0]])

In [39]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test=train_test_split(features,
                                                 label,
                                                 test_size=0.2,
                                                 random_state=10)

model=LinearRegression()
model.fit(X_train, y_train)

trainScore=model.score(X_train, y_train)
testScore=model.score(X_test, y_test)

print(f"Test Score: {testScore} and Train Score : {trainScore}")

Test Score: 0.06031275690566573 and Train Score : 0.056043915084888596


In [40]:
CL=0.05

for seed in range(1,100):

    X_train, X_test, y_train, y_test=train_test_split(features,
                                                     label,
                                                     test_size=0.2,
                                                     random_state=seed)

    model=LinearRegression()
    model.fit(X_train, y_train)

    trainScore=model.score(X_train, y_train)
    testScore=model.score(X_test, y_test)

    if testScore > trainScore and testScore >= CL:

        print(f"Test Score: {testScore} and Train Score: {trainScore} for Random Seed: {seed}")

Test Score: 0.058856880642062404 and Train Score: 0.05646667662962357 for Random Seed: 1
Test Score: 0.06309336418160005 and Train Score: 0.055490041202165385 for Random Seed: 4
Test Score: 0.06031275690566573 and Train Score: 0.056043915084888596 for Random Seed: 10
Test Score: 0.07291893678357775 and Train Score: 0.053081404455444514 for Random Seed: 11
Test Score: 0.05715105014527877 and Train Score: 0.05647895739821429 for Random Seed: 12
Test Score: 0.06822572315292863 and Train Score: 0.05391412785646543 for Random Seed: 14
Test Score: 0.06137963071779706 and Train Score: 0.05566517196924248 for Random Seed: 15
Test Score: 0.060534695099515146 and Train Score: 0.05590466143481698 for Random Seed: 16
Test Score: 0.057923133888759804 and Train Score: 0.05651484978735888 for Random Seed: 18
Test Score: 0.06548244093163469 and Train Score: 0.05455622397763682 for Random Seed: 19
Test Score: 0.06286601116656998 and Train Score: 0.05539021859109139 for Random Seed: 20
Test Score: 0.060

## This score shows that the data quality is very poor, as when we removed the features the score is reduced compared to our initial score of selecting all the features Test Score: 0.17082289846534948 and Train Score : 0.1312234795789602

### Conclusion: Feature Elimination would not serve for this dataset